# Spark Architecture and Transformations
- Notebook by Adam Lang
- Date: 9-2-2026

## Overview
- In this notebook we will go over some of the important aspects of spark architecture that is important to understand for data engineering. 
- We will also review Spark transformations.

### `explain plan` function
- The Explain Plan is a comprehensive breakdown of the logical and physical execution steps that Spark follows to process your data. Think of it as a roadmap that guides you through the inner workings of your Spark job.
- We will go over the basics of this here. 

## Load Movies table

In [0]:
## load data
df = spark.table("databricksdemo.default.movies")
df.show(3)

+--------------------+---------+------------+-----------+--------------------+-------+--------+---------+--------+--------+
|               title| industry|release_year|imdb_rating|              studio| budget| revenue|     unit|currency|language|
+--------------------+---------+------------+-----------+--------------------+-------+--------+---------+--------+--------+
|     Pather Panchali|Bollywood|        1955|        8.3|Government of Wes...|70000.0|100000.0|Thousands|     INR| Bengali|
|Doctor Strange in...|Hollywood|        2022|          7|      Marvel Studios|  200.0|   954.8| Millions|     USD| English|
|Thor: The Dark Wo...|Hollywood|        2013|        6.8|      Marvel Studios|  165.0|   644.8| Millions|     USD| English|
+--------------------+---------+------------+-----------+--------------------+-------+--------+---------+--------+--------+
only showing top 3 rows


## 1.  Spark Query with `explain` function

In [0]:
from pyspark.sql import functions as F

## query to get all movies with release_year > 2010
df_narrow = df.select("title", "studio", "imdb_rating").filter(F.col("release_year") > 2010)
df_narrow.explain("extended")

== Parsed Logical Plan ==
'Filter '`>`('release_year, 2010)
+- 'Project ['title, 'studio, 'imdb_rating]
   +- 'UnresolvedRelation [databricksdemo, default, movies], [], false

== Analyzed Logical Plan ==
title: string, studio: string, imdb_rating: string
Project [title#11384, studio#11388, imdb_rating#11387]
+- Filter (release_year#11386L > cast(2010 as bigint))
   +- Project [title#11384, studio#11388, imdb_rating#11387, release_year#11386L]
      +- SubqueryAlias databricksdemo.default.movies
         +- Relation databricksdemo.default.movies[title#11384,industry#11385,release_year#11386L,imdb_rating#11387,studio#11388,budget#11389,revenue#11390,unit#11391,currency#11392,language#11393] parquet

== Optimized Logical Plan ==
Project [title#11384, studio#11388, imdb_rating#11387]
+- Filter (isnotnull(release_year#11386L) AND (release_year#11386L > 2010))
   +- Relation databricksdemo.default.movies[title#11384,industry#11385,release_year#11386L,imdb_rating#11387,studio#11388,budget#113

### Summary
- "extended" gives us a few things as we see above

1. Analyzed logical
2. Optimized logical
3. Physical plan

- This is how it works:

```
unresolved logical plan --> SQL query on catalog --> resolved logical plan --> LOGICAL OPTIMIZER --> Optimized logical plan

```
- The physical plan is the distribution to the X number of clusters/nodes it needs to carry out the optimized query. 
- It then uses cost model + AQE (adaptive query execution) --> to optimize the best physical plan
- Then sends it to CLUSTER for final execution (viai photon executor)
- RESULTS obtained

In [0]:
## explain using "formatted"
df_narrow.explain("formatted")

== Physical Plan ==
PhotonResultStage (4)
+- PhotonColumnarToRow (3)
   +- PhotonProject (2)
      +- PhotonScan parquet databricksdemo.default.movies (1)


(1) PhotonScan parquet databricksdemo.default.movies
Output [4]: [title#11436, release_year#11438L, imdb_rating#11439, studio#11440]
DictionaryFilters: [(release_year#11438L > 2010)]
Location: PreparedDeltaFileIndex [s3://dbstorage-prod-b7paa/uc/394ac329-54a4-49c8-befe-2acb6158ed82/598229ce-7808-480e-b24e-137e565d7ccb/__unitystorage/catalogs/be641faf-722b-45f6-8a33-4e57c5337601/tables/dddb0be6-6dc8-4d93-96fe-e4148bf5e130]
ReadSchema: struct<title:string,release_year:bigint,imdb_rating:string,studio:string>
RequiredDataFilters: [isnotnull(release_year#11438L), (release_year#11438L > 2010)]

(2) PhotonProject
Input [4]: [title#11436, release_year#11438L, imdb_rating#11439, studio#11440]
Arguments: [title#11436, studio#11440, imdb_rating#11439]

(3) PhotonColumnarToRow
Input [3]: [title#11436, studio#11440, imdb_rating#11439]

(4) Pho

### Summary
- Applies filter to optimize query efficiency. 

In [0]:
display(df_narrow)

title,studio,imdb_rating
Doctor Strange in the Multiverse of Madness,Marvel Studios,7
Thor: The Dark World,Marvel Studios,6.8
Thor: Ragnarok,Marvel Studios,7.9
Thor: Love and Thunder,Marvel Studios,6.8
Interstellar,Warner Bros. Pictures,8.6
Parasite,null,8.5
Avengers: Endgame,Marvel Studios,8.4
Avengers: Infinity War,Marvel Studios,8.4
Captain America: The First Avenger,Marvel Studios,6.9
Captain America: The Winter Soldier,Marvel Studios,7.8


## 2. Spark Architecture
- The key here is there is:

1. Driver Program (spark context)
2. Cluster Manager
3. Worker Nodes (executor task, cache task)

- It looks like this:

```
Driver --> Cluster Manager --> Worker Node 1
                               Worker Node 2
                               Worker Node 3

```

In [0]:
from pyspark.sql import functions as F

## query to get all movies with release_year > 2010
df_narrow = df.select("title", "studio", "imdb_rating").filter(F.col("release_year") >= 2010)
## count rows
df_narrow.count()

21

## 3. Spark Transformations vs. Actions

### Transformation Functions (lazy evaluation)
- Analogy: Chef writing down recipe
- Defines a new dataset from existing one but does not execute yet....
```
- Select | withColumn
- filter | where
- groupBy | join | sort
- orderBy
```

### Action Functions (triggers result)
- Analogy: Chef cooking the recipe
- Action actually triggers the execution of the pending transformations and returns a result to the driver program to write or store the data.
```
- show | count
- toPandas | collect
- write | save
```

In [0]:
df = spark.table("databricksdemo.default.movies")
df.show(5,truncate=False)

+-------------------------------------------+---------+------------+-----------+-------------------------+-------+--------+---------+--------+--------+
|title                                      |industry |release_year|imdb_rating|studio                   |budget |revenue |unit     |currency|language|
+-------------------------------------------+---------+------------+-----------+-------------------------+-------+--------+---------+--------+--------+
|Pather Panchali                            |Bollywood|1955        |8.3        |Government of West Bengal|70000.0|100000.0|Thousands|INR     |Bengali |
|Doctor Strange in the Multiverse of Madness|Hollywood|2022        |7          |Marvel Studios           |200.0  |954.8   |Millions |USD     |English |
|Thor: The Dark World                       |Hollywood|2013        |6.8        |Marvel Studios           |165.0  |644.8   |Millions |USD     |English |
|Thor: Ragnarok                             |Hollywood|2017        |7.9        |Marvel S

In [0]:
## Transformation using `.filter`
from pyspark.sql import functions as F

## query to get all movies with release_year > 2010
df_narrow = df.select("title", "studio", "imdb_rating").filter(F.col("release_year") >= 2010)

display(df_narrow)

title,studio,imdb_rating
Doctor Strange in the Multiverse of Madness,Marvel Studios,7
Thor: The Dark World,Marvel Studios,6.8
Thor: Ragnarok,Marvel Studios,7.9
Thor: Love and Thunder,Marvel Studios,6.8
Interstellar,Warner Bros. Pictures,8.6
Parasite,null,8.5
Avengers: Endgame,Marvel Studios,8.4
Avengers: Infinity War,Marvel Studios,8.4
Captain America: The First Avenger,Marvel Studios,6.9
Captain America: The Winter Soldier,Marvel Studios,7.8


### Transformation Execution
- nothing actually happens due to "lazy evaluation" until you show or display that is the actual execution

In [0]:
from pyspark.sql.functions import col

## Transformations -- nothing happens just yet
df = spark.read.csv("data.csv") ## transformation 1
df_filtered = df.filter(col("age") > 18) ## transformation 2
df_selected = df_filtered.select("name") ## transformation 3

## Action -- how spark executes the transformations
df_selected.show() ## Action -- trigers all above transformations

## Why do we use "lazy evaluation"?

### 1. Query Optimization
- This allows us to instead of running 3 separate Spark operations in a row as we see below (e.g. filter -> select -> filter on !=), instead we can do this:
    - combine both filters into one
    - push the filters down to the data source (e.g. instead of 1billion records can filter directly down to the collection you need such as 10 records)
    - only read the `name` and `age` columns rather than iterating through all columns 

In [0]:
df.filter(col("age") > 18).select("name").filter(col("name") != "")

### 2. Avoid Unnecessary Work

In [0]:
## build complex transformations
big_result = df.join(df2).groupBy("category").sum("amount")

## but only need the first 5 rows
big_result.show(5) ## Action -- spark can optimize to stop filtering after only finding 5 rows

### 3. Memory Efficient
- The opposite of lazy execution is **Eager Execution** which is step by step execution such as this:
```
Step 1: Read 1GB into df1 -> Store 1GB
Step 2: Filter into df2 -> Store 500MB
Step 3: Select into df3 -> Store 200MB
Total memory: 1.7GB
-- total 3 dataframes ---
```
- However, if we use **Lazy Execution** it allows us to process data in an efficient pipeline:

```
Process data in pipeline: 1GB -> filter -> select -> result
Total memory: Much less (only final result)


```

### 4. Fault Tolerant
- Allows for planning most efficient driver -- node execution

# Narrow vs. Wide Transformations

## 1. Narrow Transformations
- this allows us to narrow down the filter immediately to find the result
- Each output partition depends upon a single input partition -- no shuffling occurs (e.g. select, filter, withColumn)

- Narrow Transformation functions include:

```
Select | withColumn
filter | where

```

In [0]:
## narrow transformation example 1

## load table
df = spark.table("databricksdemo.default.movies")

## filter
df_narrow = df.select("title", "studio", "imdb_rating").filter(F.col("release_year") >= 2010)

## count rows
df_narrow.count()

21

In [0]:
## narrow transformation example 2
df.filter(F.col("age") > 18)
df.select("name", "age")
df.withColumn("doubled_salary", F.col("salary") * 2)

## 2. Wide Transformations
- Output partitions depend upon MULTIPLE input partitions -- which requires shuffling/Exchange between stages of functions (e.g. groupBy, non-broadcast join, orderBy)
- Wide transformation functions include:

```
groupBy | join | sort
orderBy

```

In [0]:
## wide example 1
spark.conf.set("spark.sql.shuffle.partitions", 4)

## wide partition 1
wide_1 = (
    df.where(F.col("release_year") >= 2010)
      .repartition(4)
      .groupBy("studio")
      .agg(F.round(F.avg(F.col("revenue").cast("double")), 2).alias("avg_revenue"))
)
wide_1.explain("formatted")

== Physical Plan ==
AdaptiveSparkPlan (14)
+- == Initial Plan ==
   PhotonResultStage (13)
   +- PhotonColumnarToRow (12)
      +- PhotonGroupingAgg (11)
         +- PhotonShuffleExchangeSource (10)
            +- PhotonShuffleMapStage (9)
               +- PhotonShuffleExchangeSink (8)
                  +- PhotonGroupingAgg (7)
                     +- PhotonShuffleExchangeSource (6)
                        +- PhotonShuffleMapStage (5)
                           +- PhotonShuffleExchangeSink (4)
                              +- PhotonSort (3)
                                 +- PhotonProject (2)
                                    +- PhotonScan parquet databricksdemo.default.movies (1)


(1) PhotonScan parquet databricksdemo.default.movies
Output [3]: [release_year#11903L, studio#11905, revenue#11907]
DictionaryFilters: [(release_year#11903L >= 2010)]
Location: PreparedDeltaFileIndex [s3://dbstorage-prod-b7paa/uc/394ac329-54a4-49c8-befe-2acb6158ed82/598229ce-7808-480e-b24e-137e565d7ccb/

In [0]:
## wide transformation example 2
df.groupBy("department").sum("salary")
df.orderBy("name")
df1.join(df2, "customer_id")
df.distinct()

# Partitions and Parallelism
- Partitions is a unit of parallelism.

## `repartition`
- spark partitions a dataframe across multiple nodes (e.g. Node A, Node B, Node C)

In [0]:
revenue_df = df.groupBy("studio").agg(F.avg(F.col("revenue").cast("double")))

- So when we use repartition this makes this more efficient
- this allows us to efficiently aggregate the data on invididual nodes before bringing it together

In [0]:
## this will put records for each "studio" on a single node
## example: marvel 1 node, universal 1 node, etc.
base = df.repartition(6, "studio")
revenue_df = base.groupBy("studio").agg(F.avg(F.col("revenue").cast("double")))

- This is also useful when performing multiple operations as below. 
- A window in PySpark is a tool that lets you perform calculations across a specific group of related rows while keeping every original row in your DataFrame.      

    - Unlike a standard groupBy() which collapses your rows into a single summary row, a window function calculates a new value for each individual row based on its relationship to other rows.

## `Window`

In [0]:
from pyspark.sql import Window

# One-time shuffle:
base = df.repartition(6, "studio")

## reuse partitioning on same detailed rows:
agg = base.groupBy("studio").agg(F.avg(F.col("revenue").cast("double")))
ranked = base.withColumn("rnk", F.row_number().over(Window.partitionBy("studio").orderBy(F.desc("revenue"))))